In [1]:
from numpy.random import random
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

# 20newsgroups por ser un dataset clásico de NLP ya viene incluido y formateado
# en sklearn
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

In [3]:
# cargamos los datos (ya separados de forma predeterminada en train y test)
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

In [4]:
# instanciamos un vectorizador
# ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
tfidfvect = TfidfVectorizer()

In [5]:
# en el atributo `data` accedemos al texto
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


In [6]:
# con la interfaz habitual de sklearn podemos fitear el vectorizador
# (obtener el vocabulario y calcular el vector IDF)
# y transformar directamente los datos
X_train = tfidfvect.fit_transform(newsgroups_train.data)
# `X_train` la podemos denominar como la matriz documento-término

In [7]:
# recordar que las vectorizaciones por conteos son esparsas
# por ello sklearn convenientemente devuelve los vectores de documentos
# como matrices esparsas
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


In [8]:
# una vez fiteado el vectorizador, podemos acceder a atributos como el vocabulario
# aprendido. Es un diccionario que va de términos a índices.
# El índice es la posición en el vector de documento.
tfidfvect.vocabulary_['car']

25775

In [9]:
# es muy útil tener el diccionario opuesto que va de índices a términos
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

In [10]:
# en `y_train` guardamos los targets que son enteros
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

In [11]:
# hay 20 clases correspondientes a los 20 grupos de noticias
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

In [12]:
# Veamos similaridad de documentos. Tomemos algún documento
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

In [13]:
# midamos la similaridad coseno con todos los documentos de train
cossim = cosine_similarity(X_train[idx], X_train)[0]

In [14]:
# podemos ver los valores de similaridad ordenados de mayor a menos
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

In [15]:
# y a qué documentos corresponden
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  4703, 10870,  4333])

In [16]:
# los 5 documentos más similares:
mostsim = np.argsort(cossim)[::-1][1:6]

In [17]:
# el documento original pertenece a la clase:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

In [18]:
# y los 5 más similares son de las clases:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

In [19]:
# es muy fácil instanciar un modelo de clasificación Naïve Bayes y entrenarlo con sklearn
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

In [20]:
# con nuestro vectorizador ya fiteado en train, vectorizamos los textos
# del conjunto de test
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

In [21]:
# el F1-score es una metrica adecuada para reportar desempeño de modelos de claificación
# es robusta al desbalance de clases. El promediado 'macro' es el promedio de los
# F1-score de cada clase. El promedio 'micro' es equivalente a la accuracy que no
# es una buena métrica cuando los datasets son desbalanceados
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

### Consigna del desafío 1

**1**. Vectorizar documentos. Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

In [22]:
# Get the indices of 5 random documents
import random
random_indices = random.sample(range(len(newsgroups_train.data)), 5)
random_indices = [6590, 3235, 7060, 2328, 10821]

# Get the corresponding documents
random_documents = [newsgroups_train.data[i] for i in random_indices]

In [23]:
# midamos la similaridad coseno con todos los documentos de train
cossim = cosine_similarity(X_train[random_indices], X_train)

In [24]:
sorted_similarities = np.sort(cossim, axis=1)[:, ::-1]

In [26]:
higher_similarities = sorted_similarities[:, :5]

In [27]:
higher_similarities

array([[1.        , 0.29709751, 0.27638833, 0.23668244, 0.20495522],
       [1.        , 0.24332264, 0.23862796, 0.23072737, 0.22732543],
       [1.        , 0.15532832, 0.14561416, 0.14316615, 0.14266601],
       [1.        , 0.43911416, 0.43257001, 0.42492059, 0.42219765],
       [1.        , 0.38380582, 0.18141143, 0.16468738, 0.16387756]])

In [28]:

most_similar_docs_indices = np.argsort(cossim)[:,::-1]

In [29]:
most_similar_docs_indices = most_similar_docs_indices[:, :5]

In [30]:
most_similar_docs_indices

array([[ 6590,  7584,  5550,  9905,  6894],
       [ 3235,  3433,  1497, 11254,  1368],
       [ 7060,  5856,  5324, 10575,  6473],
       [ 2328,  6700, 10836,  8932, 10229],
       [10821,  3591,  4230,  4271,  1277]])

In [31]:
similar_news_0 = [newsgroups_train.data[index] for index in most_similar_docs_indices[0,:]]
similar_news_1 = [newsgroups_train.data[index] for index in most_similar_docs_indices[1,:]]
similar_news_2 = [newsgroups_train.data[index] for index in most_similar_docs_indices[2,:]]
similar_news_3 = [newsgroups_train.data[index] for index in most_similar_docs_indices[3,:]]
similar_news_4 = [newsgroups_train.data[index] for index in most_similar_docs_indices[4,:]]

### Grupo de documentos 0

In [32]:
for element in similar_news_0:
    print(element)
    print('============================================================')

[...]
[...]

I just found out from my source that this article was a joke.  Heh heh..  
It seemed pretty damn convincing to me from the start -- I just didn't
notice the smiley at the end of the article, and there were a few other
hints which I should of caught.

Anyway -- I guess this 'joke' did turn out to resemble Clinton's true 
feelings at least to some extent.  

Sorry about that...
[article deleted]

It sounds like a joke (but then the war on drugs has always been a joke...).


Heh, heh, heh, heh....I laugh because I have the same damn TV, and it
did the same thing!  Actually it is a Goldstar, but it's essentially the
same TV and electronics--just a different face plate and name.

#1.  Fortunately, TV tubes don't explode.  I'd think the TV mfrs want
to make this possibility remote as possible.  If at all, they'll 
*implode* and the glass that blows out would be the result of the
glass boucing off the back of the tube due to the implosion. In any
case, don't kick it around! :-) 


Algunos documentos paraecen estar mas relacionados que otros. Varios nombran a presidentes o hablan de entrevistas de TV, de electronica. 

In [33]:
for i in most_similar_docs_indices[0,:]:
    print(newsgroups_train.target_names[y_train[i]]) 

sci.crypt
sci.crypt
sci.electronics
sci.med
talk.politics.guns


### Grupo de documentos 1

In [34]:
for element in similar_news_1:
    print(element)
    print('============================================================')




No, he gives the keys to the FBI (who may then give them to the local police
on request) who then simply put some alagator clips on your phone junction
box and conduct an illegal tap. They then decrypt when they recover the tape.
Its just doing what the government does best: breaking the law.
It might pay to start looking at what this proposal might mean to a
police agency.  It just might be a bad idea for them, too.

OK, suppose the NY State Police want to tap a suspect's phone.  They
need a warrant, just like the old days.  But unlike the old days, they
now need to 

   (a) get two federal agencies to give them the two parts of
       the key.

Now, what happens if there's a tiff between the two escrow houses?
Posession/release of keys becomes a political bargaining chit.  State
and lower-level police agencies have to watch the big boys play politics,
while potentially good leads disappear, lives and property are lost,
statutes of limitations run out, etc.  Not to mention: a moder

el tema que se repite en esta serie de artticulos es que se nombra al FBI. Ademas se habla de seguridad criptografica.

In [35]:
for i in most_similar_docs_indices[1,:]:
    print(newsgroups_train.target_names[y_train[i]]) 

sci.crypt
sci.crypt
sci.crypt
sci.crypt
sci.crypt


### Grupo de documentos 2

In [36]:
for element in similar_news_2:
    print(element)
    print('============================================================')



Would the sub-orbital version be suitable as-is (or "as-will-be") for use
as a reuseable sounding rocket?



Thank Ghod! I had thought that Spacelifter would definitely be the
bastard Son of NLS.


(And just as a reminder:)

Thanks for posting this and making it available. This post will be LONG, I will
comment on most of it, and am reluctantly leaving all of the original in place
to provide context.

Please note that an alt. group has been set up for the Clipper stuff.

                                                     ^^^^^^^^^
Hum, AT&T, VLSI and Mykotronx are 'industry'?
Wonder what happened to IBM, this should be right up their street.
And a mandateed scheme is voluntary? Mr Orwell would love this.

                                                 ^^^^^^^^^

Telephone encryption and scrambleing are years behind digital ones like RSA,
IDEA, or even DES. The above, while literaly true, is a clasic straw-man claim
in the context of non-real-time circuits such as E-mail and the l

en este grupo hay dos temas entremezclados, tecnologia espacial , seguridad de la informacion y organismos de seguridad gubernamentales

In [37]:
for i in most_similar_docs_indices[2,:]:
    print(newsgroups_train.target_names[y_train[i]]) 

sci.space
sci.crypt
sci.space
sci.crypt
sci.space


### Grupo de documentos 3

In [38]:
for element in similar_news_3:
    print(element)
    print('============================================================')

: I
: |> Jim,
: |> 
: |> I always thought that homophobe was only a word used at Act UP
: |> rallies, I didn't beleive real people used it. Let's see if we agree
: |> on the term's definition. A homophobe is one who actively and
: |> militantly attacks homosexuals because he is actually a latent
: |> homosexual who uses his hostility to conceal his true orientation.
: |> Since everyone who disapproves of or condemns homosexuality is a
: |> homophobe (your implication is clear), it must necessarily follow that
: |> all men are latent homosexuals or bisexual at the very least.
: |> 
: 
: Crap crap crap crap crap.  A definition of any type of 'phobe comes from
: phobia = an irrational fear of.  Hence a homophobe (not only in ACT UP meetings,
: the word is apparently in general use now.  Or perhaps it isn't in the bible?  
: Wouldst thou prefer if I were to communicate with thou in bilespeak?)
: 
: Does an arachnophobe have an irrational fear of being a spider?  Does an
: agoraphobe have a

En este grupo si hay bastante coherencia, todos los documentos tratan sobre sexualidad y/ o religion.

In [39]:
for i in most_similar_docs_indices[3,:]:
    print(newsgroups_train.target_names[y_train[i]]) 

alt.atheism
soc.religion.christian
alt.atheism
alt.atheism
talk.religion.misc


### Grupo de documentos 4

In [40]:
for element in similar_news_4:
    print(element)
    print('============================================================')

I've been reading, with much confusion, about whether or not to use
ATManager. Lately, all the packages I've been buying have all
included ATManager as a "bonus"
I do some desktop publishing using PageMaker and Coreldraw.
Coreldraw comes with a nifty laser disk that contains over 200 diff
types. Add that to the TTfonts that come with win31 and you have a
decent amount of fonts. I print my creations out on an HP4
Postcript, at 600 dpi resolution with the "Resolution Enhancement 
Technology" and ..  well ... I get some darn good copies. 
So good that there isn't any diff whether or not ATManager is turned
on or not. Is it worth it to run ATM at all? Especially with these
better printer technologies ... and TT?

   >>So good that there isn't any diff whether or not ATManager is turned
   >>on or not. Is it worth it to run ATM at all? Especially with these
   >>better printer technologies ... and TT?
   >
   >There are some fonts that are only available as PS fonts.  If you
   >have a PS f

Estos textos son una mezcla de hardware/impresoras / caracteristicas de impresiones y uina entrevista muy larga entre presidente y secretario donde hablan de educacion enttre otros temas.

In [41]:
for i in most_similar_docs_indices[4,:]:
    print(newsgroups_train.target_names[y_train[i]]) 

comp.os.ms-windows.misc
comp.os.ms-windows.misc
comp.os.ms-windows.misc
talk.politics.misc
comp.sys.ibm.pc.hardware


Como conclusion general, a veces los 5 documentos mas relacionados con uno determinado tienen muchas caracteristicas semanticas en comun, otras quizas solo expresiones , o formato. no siempre las cattegorias matchean en todos.
Esto se podria explicar con los valores de similaridad resultantes, que en algunos grupos son mas altos que en otros.


**2**. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación
(f1-score macro) en el conjunto de datos de test. Considerar cambiar parámteros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial
y ComplementNB.


**3**. Transponer la matriz documento-término. De esa manera se obtiene una matriz
término-documento que puede ser interpretada como una colección de vectorización de palabras.
Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares. **La elección de palabras no debe ser al azar para evitar la aparición de términos poco interpretables, elegirlas "manualmente"**.
